# Lab Day 1 — Nguyễn Văn Giáp — 2A202602903

Notebook chạy trọn Part 0–4. Chọn cấu hình bằng validation; eval chỉ chấm cuối. Chạy từ repo, thư mục nộp hoặc code/. Trên Colab/Kaggle, đặt repo (gồm bài nộp, data/, scripts/) trong working directory; cài requirements.txt nếu cần. CPU hoặc CUDA, BF16 thí nghiệm theo thiết bị hiện tại; FP16 cần GPU và không được đo ở lần chạy CPU này.

In [1]:
import sys, subprocess, json
from pathlib import Path
import numpy as np
import torch
for base in (Path.cwd(), *Path.cwd().parents):
    candidate = base / 'submission_2A202602903' / 'code'
    if (candidate / 'lab.ipynb').is_file():
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError('Run from the repository or submission_2A202602903/code/')
sys.path.insert(0, str(CODE_DIR))
REPO = CODE_DIR.parents[1]
OUT = CODE_DIR.parent
for name in ('figures', 'results'):
    (OUT / name).mkdir(exist_ok=True)
torch.set_num_threads(2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from data import prepare_data
from train import DEFAULT_CFG
from lab_workflow import health_checks, run_group, describe_group, finish, PREDICTIONS
print('PyTorch', torch.__version__, '| device:', device, '| output:', OUT)

PyTorch 2.14.1+cpu | device: cpu | output: /home/giap/Desktop/Workspace/Vin AI/Day16/K4-Track4-Day1-NguyenVanGiap-2A202602903-Neuralnetwork/submission_2A202602903


/home/giap/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-sr0388pn because there was an issue with the default path (/home/giap/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Part 0 — Dữ liệu
Split train/eval theo metadata; validation phân tầng seed 42. Chuẩn hoá chỉ fit train, giữ 44 cột one-hot.

In [2]:
if not all((REPO / 'data/processed' / f'{split}.npz').exists() for split in ('train', 'eval')):
    subprocess.run([sys.executable, str(REPO / 'scripts/split_data.py')], cwd=REPO, check=True)
data = prepare_data(device, val_fraction=.2, seed=42, processed_dir=str(REPO / 'data/processed'))
assert data['X_tr'].shape == (371847, 54)
assert data['X_val'].shape == (92962, 54)
assert data['X_eval'].shape == (116203, 54)
assert torch.allclose(data['X_tr'][:,:10].mean(0), torch.zeros(10, device=device), atol=1e-4)
assert torch.allclose(data['X_tr'][:,:10].std(0, unbiased=False), torch.ones(10, device=device), atol=1e-4)
print('Train-only mean/std checks passed; binary columns unchanged.')

Training set size: 371847
Validation set size: 92962
Majority class: 1, Accuracy on validation set: 0.4876
Train-only mean/std checks passed; binary columns unchanged.


## Part 1 — Model và sức khoẻ
Kiểm tra số tham số, shape, loss bước 0, gradient từng tham số và quá khớp 20 mẫu trong 400 update. Loss ln 7 chỉ là mốc khi logit gần đồng đều; He ở lớp ra có thể tạo độ lệch.

In [3]:
health = health_checks(data, OUT)

{
  "params": 47879,
  "logits_shape": [
    8,
    7
  ],
  "step0_loss": 2.269061832065229,
  "ln7": 1.9459101490553132,
  "overfit20_final_loss": 5.316702299751341e-06,
  "gradient_norms": {
    "net.0.weight": 0.5070549845695496,
    "net.0.bias": 0.3422401249408722,
    "net.2.weight": 2.0212278366088867,
    "net.2.bias": 0.44387364387512207,
    "net.4.weight": 1.9604557752609253,
    "net.4.bias": 0.5724555253982544
  },
  "train_numeric_mean": [
    -2.0484735330228432e-07,
    -3.816266769263166e-08,
    5.810574066344998e-08,
    3.03659959399738e-08,
    9.520150001662842e-09,
    1.280295958139277e-08,
    -2.108384791199569e-07,
    -3.6069877751288004e-07,
    -1.1797599341889509e-07,
    2.199482906917183e-08
  ],
  "train_numeric_std": [
    1.0,
    0.9999999403953552,
    1.0,
    1.0,
    0.9999999403953552,
    1.0,
    0.9999999403953552,
    1.0,
    1.0,
    1.0
  ],
  "majority_val_acc": 0.4875970780849457,
  "torch_version": "2.14.1+cpu",
  "python_version": "

**Nhận xét:** loss bước 0 thực đo và gradient nằm ở output trên. Quá khớp 20 mẫu phải đạt loss < .02; assertion thất bại sẽ dừng notebook. Đường cong: ![](../figures/health_overfit20.png)

## Part 2 — Baseline và nhiễu seed
**Dự đoán trước:** SGD+momentum .05 vượt accuracy đa số; ba seed có khác biệt do khởi tạo và shuffle. Giữ baseline CE, He, batch 512, 20 epoch; lr .05 cố định trước chạy. Part 3 quét thêm lr bằng val.

In [4]:
all_results = []
bases = run_group('baseline', data, OUT, all_results)
baseline = bases[0]
seed_f1 = np.array([r['summary']['val_macro_f1'] for r in bases])
noise = 2 * seed_f1.std(ddof=1)
print('Baseline F1 mean ± sample std:', seed_f1.mean(), seed_f1.std(ddof=1), '| 2σ=', noise)
describe_group('baseline', bases, baseline, noise)

base-s1 01/20 val_loss=0.4956 F1=0.6022 (1.4s)
base-s1 02/20 val_loss=0.4298 F1=0.6918 (1.5s)
base-s1 03/20 val_loss=0.4176 F1=0.6766 (1.5s)
base-s1 04/20 val_loss=0.3794 F1=0.7320 (1.4s)
base-s1 05/20 val_loss=0.3781 F1=0.7313 (1.3s)
base-s1 06/20 val_loss=0.3415 F1=0.7535 (1.7s)
base-s1 07/20 val_loss=0.3403 F1=0.7641 (1.6s)
base-s1 08/20 val_loss=0.3186 F1=0.7883 (1.7s)
base-s1 09/20 val_loss=0.2972 F1=0.7858 (1.5s)
base-s1 10/20 val_loss=0.3146 F1=0.7890 (1.3s)
base-s1 11/20 val_loss=0.2855 F1=0.8049 (1.4s)
base-s1 12/20 val_loss=0.3120 F1=0.7812 (1.4s)
base-s1 13/20 val_loss=0.2822 F1=0.8215 (1.2s)
base-s1 14/20 val_loss=0.2742 F1=0.8180 (1.3s)
base-s1 15/20 val_loss=0.2627 F1=0.8193 (1.3s)
base-s1 16/20 val_loss=0.2769 F1=0.8286 (1.4s)
base-s1 17/20 val_loss=0.2560 F1=0.8354 (1.4s)
base-s1 18/20 val_loss=0.2951 F1=0.8307 (1.5s)
base-s1 19/20 val_loss=0.2523 F1=0.8437 (1.5s)
base-s1 20/20 val_loss=0.2515 F1=0.8368 (1.5s)
base-s2 01/20 val_loss=0.4982 F1=0.6320 (1.4s)
base-s2 02/20

**Đối chiếu:** output ghi F1 từng seed, gap loss và thời gian. Ba seed chỉ ước lượng nhiễu thô; ngưỡng 2σ dùng cho các nhận xét tiếp theo, không phải kiểm định thống kê. ![](../figures/compare_baseline.png)

## Part 3 — Thí nghiệm
Cùng split, 20 epoch, batch và seed 1; Các biến thể thường Các biến thể thường đổi một yếu tố so baseline. Adam đổi optimizer và lr để mỗi optimizer có quét lr riêng; so ở lr tốt nhất đã thử. Adam đổi optimizer và lr để mỗi optimizer có quét lr riêng; so ở lr tốt nhất đã thử. Riêng clip-high đổi hai yếu tố so baseline nhưng chỉ đổi clipping so lr-high; so cặp đó để tách tác động. MSE dùng logit thô/one-hot và mean trên mẫu × 7 lớp; không so giá trị loss với CE.

### optimizer
**Dự đoán trước:** Adam cập nhật thích nghi có thể hội tụ nhanh hơn; mỗi optimizer cần thử nhiều lr.

In [5]:
subset = run_group('optimizer', data, OUT, all_results)
describe_group('optimizer', subset, baseline, noise)

opt-sgdm-lr01 01/20 val_loss=0.5797 F1=0.5136 (1.5s)
opt-sgdm-lr01 02/20 val_loss=0.5295 F1=0.5975 (1.5s)
opt-sgdm-lr01 03/20 val_loss=0.4953 F1=0.5997 (1.6s)
opt-sgdm-lr01 04/20 val_loss=0.4808 F1=0.6312 (1.5s)
opt-sgdm-lr01 05/20 val_loss=0.4488 F1=0.6276 (1.5s)
opt-sgdm-lr01 06/20 val_loss=0.4644 F1=0.6468 (1.5s)
opt-sgdm-lr01 07/20 val_loss=0.4136 F1=0.6758 (1.5s)
opt-sgdm-lr01 08/20 val_loss=0.4122 F1=0.6882 (1.5s)
opt-sgdm-lr01 09/20 val_loss=0.3976 F1=0.6803 (1.5s)
opt-sgdm-lr01 10/20 val_loss=0.3893 F1=0.7174 (1.5s)
opt-sgdm-lr01 11/20 val_loss=0.3785 F1=0.7035 (1.5s)
opt-sgdm-lr01 12/20 val_loss=0.3859 F1=0.6970 (1.5s)
opt-sgdm-lr01 13/20 val_loss=0.3656 F1=0.7355 (1.5s)
opt-sgdm-lr01 14/20 val_loss=0.3495 F1=0.7387 (1.5s)
opt-sgdm-lr01 15/20 val_loss=0.3471 F1=0.7474 (1.6s)
opt-sgdm-lr01 16/20 val_loss=0.3394 F1=0.7487 (1.5s)
opt-sgdm-lr01 17/20 val_loss=0.3395 F1=0.7213 (1.5s)
opt-sgdm-lr01 18/20 val_loss=0.3506 F1=0.7543 (1.7s)
opt-sgdm-lr01 19/20 val_loss=0.3172 F1=0.7649 

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. Momentum tích luỹ hướng gradient; Adam chia bước theo moment bậc hai. So lr tốt nhất trong phạm vi đã thử, không khẳng định tối ưu toàn cục.

![](../figures/compare_optimizer.png)

### loss
**Dự đoán trước:** MSE logit/one-hot dự kiến học chậm hơn CE vì không trực tiếp tối ưu log-likelihood.

In [6]:
subset = run_group('loss', data, OUT, all_results)
describe_group('loss', subset, baseline, noise)

loss-mse 01/20 val_loss=0.0517 F1=0.4423 (1.6s)
loss-mse 02/20 val_loss=0.0476 F1=0.4900 (1.4s)
loss-mse 03/20 val_loss=0.0452 F1=0.4961 (1.4s)
loss-mse 04/20 val_loss=0.0434 F1=0.5350 (1.5s)
loss-mse 05/20 val_loss=0.0417 F1=0.5452 (1.5s)
loss-mse 06/20 val_loss=0.0406 F1=0.5828 (1.5s)
loss-mse 07/20 val_loss=0.0394 F1=0.5861 (1.6s)
loss-mse 08/20 val_loss=0.0385 F1=0.5945 (1.5s)
loss-mse 09/20 val_loss=0.0376 F1=0.6263 (1.5s)
loss-mse 10/20 val_loss=0.0369 F1=0.6339 (1.5s)
loss-mse 11/20 val_loss=0.0361 F1=0.6383 (1.4s)
loss-mse 12/20 val_loss=0.0355 F1=0.6497 (1.5s)
loss-mse 13/20 val_loss=0.0352 F1=0.6570 (1.3s)
loss-mse 14/20 val_loss=0.0344 F1=0.6597 (1.3s)
loss-mse 15/20 val_loss=0.0343 F1=0.6722 (1.3s)
loss-mse 16/20 val_loss=0.0339 F1=0.6773 (1.4s)
loss-mse 17/20 val_loss=0.0334 F1=0.6695 (1.4s)
loss-mse 18/20 val_loss=0.0328 F1=0.6831 (1.4s)
loss-mse 19/20 val_loss=0.0327 F1=0.6840 (1.4s)
loss-mse 20/20 val_loss=0.0324 F1=0.6839 (1.4s)
Cơ chế: CE có gradient logit p-y. MSE ở 

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. CE có gradient logit p-y. MSE ở đây dùng logit thô, mean trên 7 tọa độ: gradient 2(z-onehot)/7; không so trị số loss giữa CE và MSE.

![](../figures/compare_loss.png)

### hparam
**Dự đoán trước:** lr cao .5 có thể làm loss/gradient dao động hơn .05; chưa chắc phân kỳ.

In [7]:
subset = run_group('hparam', data, OUT, all_results)
describe_group('hparam', subset, baseline, noise)

lr-high 01/20 val_loss=0.4335 F1=0.6466 (1.7s)
lr-high 02/20 val_loss=0.3970 F1=0.7208 (1.5s)
lr-high 03/20 val_loss=0.3719 F1=0.7220 (1.7s)
lr-high 04/20 val_loss=0.3616 F1=0.7492 (1.6s)
lr-high 05/20 val_loss=0.3321 F1=0.7182 (1.7s)
lr-high 06/20 val_loss=0.3243 F1=0.7972 (1.5s)
lr-high 07/20 val_loss=0.3105 F1=0.7867 (1.5s)
lr-high 08/20 val_loss=0.2921 F1=0.8141 (1.5s)
lr-high 09/20 val_loss=0.2894 F1=0.7772 (1.5s)
lr-high 10/20 val_loss=0.2802 F1=0.8201 (1.5s)
lr-high 11/20 val_loss=0.2752 F1=0.8107 (1.6s)
lr-high 12/20 val_loss=0.2684 F1=0.8138 (1.5s)
lr-high 13/20 val_loss=0.2630 F1=0.8290 (1.6s)
lr-high 14/20 val_loss=0.2639 F1=0.8256 (1.5s)
lr-high 15/20 val_loss=0.2656 F1=0.8303 (1.5s)
lr-high 16/20 val_loss=0.2814 F1=0.8261 (1.5s)
lr-high 17/20 val_loss=0.2735 F1=0.8285 (1.5s)
lr-high 18/20 val_loss=0.2650 F1=0.8295 (1.6s)
lr-high 19/20 val_loss=0.2716 F1=0.8212 (1.6s)
lr-high 20/20 val_loss=0.2603 F1=0.8299 (1.6s)
Cơ chế: Tăng lr làm bước cập nhật lớn hơn; cùng 20 epoch và 

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. Tăng lr làm bước cập nhật lớn hơn; cùng 20 epoch và batch 512 giữ số bước như nhau.

![](../figures/compare_hparam.png)

### dropout
**Dự đoán trước:** q=.2 có thể giảm gap train–val nhưng làm chậm học nếu baseline chưa quá khớp.

In [8]:
subset = run_group('dropout', data, OUT, all_results)
describe_group('dropout', subset, baseline, noise)

dropout-q02 01/20 val_loss=0.5416 F1=0.5352 (3.3s)
dropout-q02 02/20 val_loss=0.4897 F1=0.6375 (3.3s)
dropout-q02 03/20 val_loss=0.4596 F1=0.6462 (3.2s)
dropout-q02 04/20 val_loss=0.4364 F1=0.6766 (3.2s)
dropout-q02 05/20 val_loss=0.4153 F1=0.6803 (3.1s)
dropout-q02 06/20 val_loss=0.4047 F1=0.7042 (3.4s)
dropout-q02 07/20 val_loss=0.3905 F1=0.7195 (3.3s)
dropout-q02 08/20 val_loss=0.3738 F1=0.7254 (3.5s)
dropout-q02 09/20 val_loss=0.3644 F1=0.7325 (3.3s)
dropout-q02 10/20 val_loss=0.3555 F1=0.7491 (3.2s)
dropout-q02 11/20 val_loss=0.3475 F1=0.7514 (3.0s)
dropout-q02 12/20 val_loss=0.3437 F1=0.7590 (3.0s)
dropout-q02 13/20 val_loss=0.3338 F1=0.7707 (3.9s)
dropout-q02 14/20 val_loss=0.3303 F1=0.7667 (3.5s)
dropout-q02 15/20 val_loss=0.3234 F1=0.7806 (3.4s)
dropout-q02 16/20 val_loss=0.3218 F1=0.7783 (3.1s)
dropout-q02 17/20 val_loss=0.3150 F1=0.7865 (3.0s)
dropout-q02 18/20 val_loss=0.3090 F1=0.7950 (3.0s)
dropout-q02 19/20 val_loss=0.3080 F1=0.7950 (3.1s)
dropout-q02 20/20 val_loss=0.30

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. Dropout tăng nhiễu lúc train; cả train loss và val loss đều đo eval mode. Gap cuối dương là dấu hiệu cần xét, không đủ để kết luận mọi regularization có lợi.

![](../figures/compare_dropout.png)

### clipping
**Dự đoán trước:** Cắt ở phân vị 90% gradient baseline; có thể giảm gai ở lr cao, không chắc tăng F1.

In [9]:
# Use all actual baseline step norms (across 20 epochs) to set c.
# c is the 90th percentile of all baseline step norms (no eval labels used).
clip_norm = baseline['summary']['grad_norm_p90']
print('Clipping threshold from baseline step norms p90:', clip_norm)
subset = run_group('clipping', data, OUT, all_results, clip_norm=clip_norm)
describe_group('clipping', subset, baseline, noise)
for r in subset:
    print(r['cfg']['exp_id'], 'mean clip fraction', np.mean(r['history']['clip_fraction']),
          'maximum pre-clip norm', max(r['history']['grad_norm_max']))

Clipping threshold from baseline step norms p90: 1.047024178504944
clip-normal 01/20 val_loss=0.4931 F1=0.6074 (1.5s)
clip-normal 02/20 val_loss=0.4338 F1=0.6932 (1.4s)
clip-normal 03/20 val_loss=0.4193 F1=0.6712 (1.3s)
clip-normal 04/20 val_loss=0.3798 F1=0.7341 (1.3s)
clip-normal 05/20 val_loss=0.3786 F1=0.7316 (1.3s)
clip-normal 06/20 val_loss=0.3426 F1=0.7526 (1.4s)
clip-normal 07/20 val_loss=0.3297 F1=0.7745 (1.3s)
clip-normal 08/20 val_loss=0.3426 F1=0.7830 (1.4s)
clip-normal 09/20 val_loss=0.3045 F1=0.7867 (1.5s)
clip-normal 10/20 val_loss=0.3140 F1=0.7842 (1.5s)
clip-normal 11/20 val_loss=0.2889 F1=0.7996 (1.6s)
clip-normal 12/20 val_loss=0.2801 F1=0.8046 (1.6s)
clip-normal 13/20 val_loss=0.2958 F1=0.8207 (1.6s)
clip-normal 14/20 val_loss=0.2776 F1=0.8155 (1.6s)
clip-normal 15/20 val_loss=0.2716 F1=0.8301 (1.6s)
clip-normal 16/20 val_loss=0.2628 F1=0.8257 (1.6s)
clip-normal 17/20 val_loss=0.2567 F1=0.8347 (1.6s)
clip-normal 18/20 val_loss=0.2778 F1=0.8349 (1.6s)
clip-normal 19/

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. Global L2 clipping nhân gradient với min(1,c/||g||); giữ hướng nhưng giảm bước. Nó không sửa được nhãn sai hoặc lr quá nhỏ.

![](../figures/compare_clipping.png)

### amp
**Dự đoán trước:** BF16 CPU có thể nhanh hơn nếu CPU hỗ trợ; F1 có thể lệch do làm tròn.

In [10]:
subset = run_group('amp', data, OUT, all_results)
describe_group('amp', subset, baseline, noise)

amp-bf16 01/20 val_loss=0.4977 F1=0.6084 (3.7s)
amp-bf16 02/20 val_loss=0.4317 F1=0.6926 (3.5s)
amp-bf16 03/20 val_loss=0.4217 F1=0.6803 (3.4s)
amp-bf16 04/20 val_loss=0.3744 F1=0.7361 (3.4s)
amp-bf16 05/20 val_loss=0.3808 F1=0.7381 (3.5s)
amp-bf16 06/20 val_loss=0.3452 F1=0.7553 (3.5s)
amp-bf16 07/20 val_loss=0.3322 F1=0.7776 (3.5s)
amp-bf16 08/20 val_loss=0.3259 F1=0.7947 (3.4s)
amp-bf16 09/20 val_loss=0.2995 F1=0.7880 (3.1s)
amp-bf16 10/20 val_loss=0.3197 F1=0.7856 (3.1s)
amp-bf16 11/20 val_loss=0.2937 F1=0.8045 (3.2s)
amp-bf16 12/20 val_loss=0.3140 F1=0.7780 (3.0s)
amp-bf16 13/20 val_loss=0.2773 F1=0.8224 (3.0s)
amp-bf16 14/20 val_loss=0.2825 F1=0.8181 (3.0s)
amp-bf16 15/20 val_loss=0.2640 F1=0.8262 (3.4s)
amp-bf16 16/20 val_loss=0.2710 F1=0.8209 (3.3s)
amp-bf16 17/20 val_loss=0.2590 F1=0.8315 (3.1s)
amp-bf16 18/20 val_loss=0.3109 F1=0.8274 (3.1s)
amp-bf16 19/20 val_loss=0.2525 F1=0.8402 (3.5s)
amp-bf16 20/20 val_loss=0.2660 F1=0.8380 (3.4s)
Cơ chế: Autocast BF16 chỉ bọc forward/lo

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. Autocast BF16 chỉ bọc forward/loss, tham số và optimizer vẫn FP32. CPU không có CUDA nên không đo FP16 hay CUDA peak memory; thời gian phụ thuộc phần cứng.

![](../figures/compare_amp.png)

### init
**Dự đoán trước:** Xavier có phương sai nhỏ hơn He; zeros giữ đối xứng và chỉ học bias đầu ra.

In [11]:
subset = run_group('init', data, OUT, all_results)
describe_group('init', subset, baseline, noise)

init-xavier 01/20 val_loss=0.5139 F1=0.5745 (1.6s)
init-xavier 02/20 val_loss=0.4521 F1=0.6713 (1.5s)
init-xavier 03/20 val_loss=0.4467 F1=0.6256 (1.5s)
init-xavier 04/20 val_loss=0.3960 F1=0.7063 (1.5s)
init-xavier 05/20 val_loss=0.4056 F1=0.7111 (1.5s)
init-xavier 06/20 val_loss=0.3447 F1=0.7420 (1.5s)
init-xavier 07/20 val_loss=0.3450 F1=0.7446 (1.4s)
init-xavier 08/20 val_loss=0.3186 F1=0.7759 (1.4s)
init-xavier 09/20 val_loss=0.3118 F1=0.7506 (1.4s)
init-xavier 10/20 val_loss=0.3264 F1=0.7785 (1.4s)
init-xavier 11/20 val_loss=0.2947 F1=0.7984 (1.4s)
init-xavier 12/20 val_loss=0.3013 F1=0.7805 (1.5s)
init-xavier 13/20 val_loss=0.2862 F1=0.8117 (1.5s)
init-xavier 14/20 val_loss=0.2980 F1=0.8021 (1.4s)
init-xavier 15/20 val_loss=0.2662 F1=0.8240 (1.4s)
init-xavier 16/20 val_loss=0.2731 F1=0.8135 (1.3s)
init-xavier 17/20 val_loss=0.2579 F1=0.8292 (1.3s)
init-xavier 18/20 val_loss=0.2961 F1=0.8290 (1.4s)
init-xavier 19/20 val_loss=0.2699 F1=0.8282 (1.5s)
init-xavier 20/20 val_loss=0.26

**Đối chiếu sau:** số đo output trên cho biết Δ có vượt 2σ không; Δ dương hỗ trợ cải thiện trong thiết lập này, Δ âm phản bác dự đoán cải thiện. ReLU cần phá đối xứng: zeros làm gradient các lớp ẩn bằng 0. He dùng Var=2/fan_in; Xavier dùng 2/(fan_in+fan_out). Stats đo sau từng Linear.

![](../figures/compare_init.png)

## Part 4 — Chọn cấu hình, đánh giá chính thức và bài nộp
Chọn max val macro-F1 trong các cấu hình seed 1, dùng checkpoint min val loss. Lưu selection.json trước khi chấm eval; không huấn luyện thêm sau khi xem eval. Official evaluate.py tạo nguyên bản JSON; bảng chỉ điền eval cho base-s1 và cấu hình cuối.

In [12]:
choice, baseline_scores, final_scores = finish(all_results, data, health, OUT, REPO)
print('Final config:', choice['exp_id'])
print('Eval baseline:', baseline_scores['accuracy'], baseline_scores['macro_f1'])
print('Eval final:', final_scores['accuracy'], final_scores['macro_f1'])

Selected BEFORE eval: {'exp_id': 'opt-adam-lr003', 'cfg': {'exp_id': 'opt-adam-lr003', 'group': 'optimizer', 'description': 'Adam cập nhật thích nghi có thể hội tụ nhanh hơn; mỗi optimizer cần thử nhiều lr.', 'loss': 'ce', 'optimizer': 'adam', 'lr': 0.003, 'weight_decay': 0.0, 'momentum': 0.9, 'batch': 512, 'epochs': 20, 'hidden': (256, 128), 'dropout': 0.0, 'init': 'he', 'clip_norm': None, 'precision': 'fp32', 'seed': 1}, 'selection': 'maximum val macro-F1 among seed-1 runs; checkpoint minimum val loss', 'val_macro_f1': 0.8779171445095677}
Final config: opt-adam-lr003
Eval baseline: 0.8973692589692177 0.8398580863807661
Eval final: 0.9159057855649165 0.8776600357698733


In [13]:
import pandas as pd
from IPython.display import display
print('Per-class error analysis:')
display(pd.DataFrame(final_scores['per_class']))
cm = np.array(final_scores['confusion_matrix'])
worst = min(final_scores['per_class'], key=lambda c:c['f1'])
row = cm[worst['cls']].copy(); row[worst['cls']] = 0
print('Lowest F1 class:', worst['cls'], '| most confused with:', row.argmax(), '| count:', row.max())
print('Class imbalance / overlapping terrain features are hypotheses, not measured causes.')
print('Reports, workbook and all per-experiment figures saved in', OUT)
assert len(list((OUT/'figures').glob('*.png'))) >= len(all_results)

Per-class error analysis:
Lowest F1 class: 4 | most confused with: 1 | count: 419
Class imbalance / overlapping terrain features are hypotheses, not measured causes.
Reports, workbook and all per-experiment figures saved in /home/giap/Desktop/Workspace/Vin AI/Day16/K4-Track4-Day1-NguyenVanGiap-2A202602903-Neuralnetwork/submission_2A202602903


,cls,support,precision,recall,f1
0,0,42368,0.918051,0.905093,0.911526
1,1,56661,0.917316,0.939447,0.928249
2,2,7151,0.929674,0.892882,0.910907
3,3,549,0.827094,0.845173,0.836036
4,4,1899,0.834406,0.750922,0.790466
5,5,3473,0.855867,0.827527,0.841458
6,6,4102,0.947813,0.903218,0.924978


**Phân tích lỗi:** bảng trên lấy trực tiếp JSON chính thức; ma trận và lớp khó nhất ghi trong REPORT.md. Mất cân bằng và chồng lấn địa hình là giả thuyết cần kiểm tra.

![](../figures/eval_confusion.png)

**Hạn chế:** ba seed baseline, một seed mỗi biến thể; CPU BF16 không thay thế GPU FP16; chưa đo CUDA memory. MSE dùng cùng lr baseline; phạm vi quét optimizer hẹp. Mọi kết quả cụ thể và câu hỏi dẫn dắt nằm trong [REPORT.md](../REPORT.md).